# F2-P1 private 25-record development smoke

This private Kaggle workflow reloads only the frozen F2-P1 `checkpoint-125` and generates on exactly 25 deterministically selected QALB-2014 L1 development records. It is a technical pipeline gate, not checkpoint selection or final evaluation. It does not train, tune a prompt or parser, load QALB test, load Nahw-Passage, run F3, run safety diagnostics, or activate XG. Raw responses, corpus records, and adapter bytes remain private; only a text-free audit summary may be committed.


## 1. Load one private execution configuration

Do not edit this notebook to activate inference. After the implementation PR merges, create exactly one `f2_dev_smoke_execution_config.json` with `scripts/prepare_f2_dev_smoke_execution_config.py` and attach it through a private Kaggle Dataset. Without that file, every GPU and private-data stage remains disabled.


In [ ]:
from pathlib import Path
import json, re
CONFIG_FILENAME = 'f2_dev_smoke_execution_config.json'
CONFIG_KEYS = {'stage', 'approved_workflow_commit', 'approval_reference', 'confirmation'}
DEFAULT_EXECUTION_CONFIG = {'stage': 'disabled', 'approved_workflow_commit': '', 'approval_reference': '', 'confirmation': ''}
KAGGLE_INPUT_ROOT = Path('/kaggle/input')
CONFIG_MATCHES = sorted(KAGGLE_INPUT_ROOT.rglob(CONFIG_FILENAME)) if KAGGLE_INPUT_ROOT.exists() else []
if len(CONFIG_MATCHES) > 1:
    raise RuntimeError(f'Expected at most one private {CONFIG_FILENAME}; found {len(CONFIG_MATCHES)}')
if CONFIG_MATCHES:
    EXECUTION_CONFIG = json.loads(CONFIG_MATCHES[0].read_text(encoding='utf-8'))
    if set(EXECUTION_CONFIG) != CONFIG_KEYS or not all(isinstance(value, str) for value in EXECUTION_CONFIG.values()):
        raise RuntimeError('Private execution config has an invalid schema or value type.')
    CONFIG_SOURCE = CONFIG_MATCHES[0].name
else:
    EXECUTION_CONFIG = DEFAULT_EXECUTION_CONFIG
    CONFIG_SOURCE = 'committed-disabled-defaults'
EXECUTION_STAGE = EXECUTION_CONFIG['stage']
APPROVED_WORKFLOW_COMMIT = EXECUTION_CONFIG['approved_workflow_commit']
APPROVAL_REFERENCE = EXECUTION_CONFIG['approval_reference']
CONFIRMATION_VALUE = EXECUTION_CONFIG['confirmation']
if EXECUTION_STAGE not in {'disabled', 'private-dev-smoke'}:
    raise RuntimeError('Private execution config selected an unknown stage.')
RUN_PRIVATE_DEV_SMOKE = EXECUTION_STAGE == 'private-dev-smoke'
if RUN_PRIVATE_DEV_SMOKE:
    if not re.fullmatch(r'[0-9a-f]{40}', APPROVED_WORKFLOW_COMMIT):
        raise RuntimeError('Private execution config has an invalid approved commit.')
    if not re.fullmatch(r'https://github\.com/ALBA7OOTH-Research-Lab/Musahhih/issues/82#issuecomment-[0-9]+', APPROVAL_REFERENCE):
        raise RuntimeError('Private execution config has an invalid approval reference.')
    if CONFIRMATION_VALUE != 'RUN_F2_P1_PRIVATE_DEV_SMOKE_25':
        raise RuntimeError('Exact private development smoke confirmation is missing.')
print({'config_source': CONFIG_SOURCE, 'stage': EXECUTION_STAGE, 'approved_workflow_commit': APPROVED_WORKFLOW_COMMIT or None})


## 2. Safe immutable repository setup

An executing run fetches and detach-checks out the exact authorized workflow commit before verification. A dirty or unexpected repository fails closed; no user work is deleted.


In [ ]:
import os, subprocess, sys
RUNTIME_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
REPO = RUNTIME_ROOT / 'Musahhih'
REPO_URL = 'https://github.com/ALBA7OOTH-Research-Lab/Musahhih.git'
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
origin_url = subprocess.run(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], capture_output=True, text=True, check=True).stdout.strip()
if origin_url != REPO_URL:
    raise RuntimeError(f'Unexpected repository origin: {origin_url}')
dirty = subprocess.run(['git', '-C', str(REPO), 'status', '--porcelain'], capture_output=True, text=True, check=True).stdout
if dirty and RUN_PRIVATE_DEV_SMOKE:
    raise RuntimeError('Executing stage refuses a repository with local changes.')
if dirty:
    print('Existing repository has local changes; leaving it unchanged.')
elif RUN_PRIVATE_DEV_SMOKE:
    subprocess.run(['git', '-C', str(REPO), 'fetch', '--no-tags', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', APPROVED_WORKFLOW_COMMIT], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
os.chdir(REPO)
ACTUAL_WORKFLOW_COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
if RUN_PRIVATE_DEV_SMOKE and ACTUAL_WORKFLOW_COMMIT != APPROVED_WORKFLOW_COMMIT:
    raise RuntimeError('Approved workflow commit mismatch.')
print({'repository': str(REPO), 'git_sha': ACTUAL_WORKFLOW_COMMIT})


## 3. Validate private checkpoint and development records

Attach the private F2/F3 record Dataset and the completed private F2-P1 training-kernel output. This stage locates exact expected filenames, validates counts and hashes, and prints aggregate metadata only. It never prints Arabic corpus text.


In [ ]:
import hashlib
from scripts.f2_f3_training_utils import *
from scripts.f1_training_utils import sha256_file
EXPECTED_ADAPTER_MODEL_SHA256 = '935fdf02c95189934e40629f877d8692d325ef22895cbaa03fdb7390b0cd7b3e'
EXPECTED_ADAPTER_CONFIG_SHA256 = 'b07ab34155647961ea1de8fbfff0db8e17d00229da01f2b941a15a78499da986'
EXPECTED_SELECTION_SHA256 = '39edee5e31d79c791a4ab0b14b7b85b838e28bcc302d9e552f168a03ac870e1b'
EXPECTED_WORKFLOW_COMMIT = 'f64edead0367e7659b107e5c4c309ed811d09071'
PRIVATE_INPUT_SUMMARY = {'executed': False, 'contains_corpus_text': False}
if RUN_PRIVATE_DEV_SMOKE:
    selection_candidates = []
    for candidate in KAGGLE_INPUT_ROOT.rglob('checkpoint_selection.json'):
        try:
            payload = json.loads(candidate.read_text(encoding='utf-8'))
        except (OSError, UnicodeDecodeError, json.JSONDecodeError):
            continue
        if payload.get('arm') == 'F2-P1' and payload.get('workflow_commit') == EXPECTED_WORKFLOW_COMMIT:
            selection_candidates.append((candidate, payload))
    if len(selection_candidates) != 1:
        raise RuntimeError(f'Expected one private F2-P1 checkpoint selection artifact; observed {len(selection_candidates)}')
    CHECKPOINT_SELECTION_PATH, CHECKPOINT_SELECTION = selection_candidates[0]
    if sha256_file(CHECKPOINT_SELECTION_PATH) != EXPECTED_SELECTION_SHA256:
        raise RuntimeError('Private checkpoint-selection SHA-256 mismatch.')
    if CHECKPOINT_SELECTION.get('selected_checkpoint') != 'checkpoint-125' or len(CHECKPOINT_SELECTION.get('evaluations', [])) != 2:
        raise RuntimeError('Private checkpoint selection does not match the frozen F2-P1 run.')
    PRIVATE_RUN_DIR = CHECKPOINT_SELECTION_PATH.parent
    SELECTED_CHECKPOINT = PRIVATE_RUN_DIR / CHECKPOINT_SELECTION['selected_checkpoint']
    ADAPTER_MODEL_PATH = SELECTED_CHECKPOINT / 'adapter_model.safetensors'
    ADAPTER_CONFIG_PATH = SELECTED_CHECKPOINT / 'adapter_config.json'
    if not SELECTED_CHECKPOINT.is_dir() or not ADAPTER_CONFIG_PATH.is_file() or not ADAPTER_MODEL_PATH.is_file():
        raise RuntimeError('Selected private checkpoint is incomplete.')
    if sha256_file(ADAPTER_MODEL_PATH) != EXPECTED_ADAPTER_MODEL_SHA256:
        raise RuntimeError('Selected adapter-model SHA-256 mismatch.')
    if sha256_file(ADAPTER_CONFIG_PATH) != EXPECTED_ADAPTER_CONFIG_SHA256:
        raise RuntimeError('Selected adapter-config SHA-256 mismatch.')
    adapter_config = json.loads(ADAPTER_CONFIG_PATH.read_text(encoding='utf-8'))
    if adapter_config.get('r') != 16 or adapter_config.get('lora_alpha') != 32 or adapter_config.get('lora_dropout') != 0.0:
        raise RuntimeError('Selected adapter configuration does not match the frozen LoRA contract.')
    DEV_PATH = locate_private_input('common_dev_records.jsonl', KAGGLE_INPUT_ROOT, Path('data/processed/f2_f3_training_records'))
    DEV_INPUT = validate_private_records(DEV_PATH, 'development')
    with DEV_PATH.open('r', encoding='utf-8') as stream:
        private_dev = [json.loads(line) for line in stream]
    PRIVATE_INPUT_SUMMARY = {'executed': True, 'checkpoint': SELECTED_CHECKPOINT.name, 'adapter_model_sha256': EXPECTED_ADAPTER_MODEL_SHA256, 'checkpoint_selection_sha256': EXPECTED_SELECTION_SHA256, 'adapter_config_sha256': EXPECTED_ADAPTER_CONFIG_SHA256, 'development': DEV_INPUT, 'contains_corpus_text': False}
print(json.dumps(PRIVATE_INPUT_SUMMARY, indent=2))


## 4. P100 runtime and dependency preflight

The authorized environment is one free private Kaggle NVIDIA P100 with Internet enabled and the pinned original image. CPU is not an inference fallback. Package installation and CUDA imports remain untouched while execution is disabled. The conditional setup reuses an already compatible frozen stack and otherwise performs one pinned CUDA 12.4 installation.


In [ ]:
RUNTIME = {'executed': False}
if RUN_PRIVATE_DEV_SMOKE:
    assigned_gpu_name = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True, check=True).stdout.strip()
    if len([line for line in assigned_gpu_name.splitlines() if line.strip()]) != 1 or 'P100' not in assigned_gpu_name:
        raise RuntimeError('This frozen workflow requires exactly one Kaggle NVIDIA P100; CPU is not an inference fallback.')
    P100_STACK = dict(P100_CORE_STACK) | {'index': 'https://download.pytorch.org/whl/cu124'}
    import importlib.metadata as md, time
    def installed_package_versions(names):
        versions = {}
        for name in names:
            try:
                versions[name] = md.version(name)
            except md.PackageNotFoundError:
                versions[name] = None
        return versions
    INITIAL_P100_STACK = installed_package_versions(P100_CORE_STACK)
    INITIAL_HEAVY_REPORT = p100_stack_report(INITIAL_P100_STACK, P100_HEAVY_STACK)
    HEAVY_STACK_INSTALL_SECONDS = 0.0
    if not INITIAL_HEAVY_REPORT['compatible']:
        print('Installing the frozen P100 heavy stack once; progress bars are disabled.')
        started = time.monotonic()
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--progress-bar', 'off', '--upgrade', f"torch=={P100_STACK['torch']}", f"torchvision=={P100_STACK['torchvision']}", '--index-url', P100_STACK['index']], check=True)
        if 'numpy' in INITIAL_HEAVY_REPORT['mismatches']:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--progress-bar', 'off', '--upgrade', f"numpy=={P100_STACK['numpy']}"], check=True)
        HEAVY_STACK_INSTALL_SECONDS = round(time.monotonic() - started, 3)
    POST_HEAVY_STACK = installed_package_versions(P100_CORE_STACK)
    POST_HEAVY_REPORT = validate_p100_core_stack(POST_HEAVY_STACK, P100_HEAVY_STACK)
    os.environ['UNSLOTH_COMPILE_DISABLE'] = '1'
    import torch
    if not torch.cuda.is_available() or torch.cuda.device_count() != 1:
        raise RuntimeError('Exactly one CUDA GPU is required.')
    if torch.version.cuda != '12.4':
        raise RuntimeError(f'Expected CUDA 12.4, found {torch.version.cuda!r}.')
    gpu = torch.cuda.get_device_properties(0)
    torch.ones(1, device='cuda').sum().item(); torch.cuda.synchronize()
    print({'gpu': gpu.name, 'vram_gib': round(gpu.total_memory / 1024**3, 2)})
else:
    print('Private development smoke disabled: CUDA and package installation were not touched.')


In [ ]:
if RUN_PRIVATE_DEV_SMOKE:
    def pip_install(*items):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--progress-bar', 'off', *items], check=True)
    pip_install('sentencepiece', 'protobuf', 'datasets==4.3.0', 'huggingface_hub>=0.34.0', 'hf_transfer')
    pip_install('--no-deps', 'unsloth_zoo', 'bitsandbytes', 'accelerate', f"xformers=={P100_STACK['xformers']}", 'peft', 'trl', 'triton', 'unsloth')
    pip_install('--no-deps', '--upgrade', 'torchao==0.16.0')
    pip_install('transformers==4.56.2')
    pip_install('--no-deps', 'trl==0.22.2')
    FINAL_STACK_REPORT = validate_p100_core_stack(installed_package_versions(P100_CORE_STACK))
    import platform
    packages = ['torch', 'transformers', 'unsloth', 'accelerate', 'peft', 'trl', 'datasets', 'bitsandbytes']
    VERSIONS = {name: md.version(name) for name in packages}
    RUNTIME = {'executed': True, 'python': platform.python_version(), 'cuda': torch.version.cuda, 'gpu': gpu.name, 'cuda_capability': f'{gpu.major}.{gpu.minor}', 'visible_gpu_count': torch.cuda.device_count(), 'heavy_install_performed': not INITIAL_HEAVY_REPORT['compatible'], 'heavy_install_seconds': HEAVY_STACK_INSTALL_SECONDS, 'core_stack': FINAL_STACK_REPORT, 'packages': VERSIONS}
    print(json.dumps(RUNTIME, indent=2))


## 5. Reload the immutable selected adapter

Load the selected checkpoint unmerged in 4-bit mode and switch to inference. No fresh LoRA, trainer, optimizer, or weight update is created.


In [ ]:
MODEL_INFO = {'executed': False}
if RUN_PRIVATE_DEV_SMOKE:
    from transformers import set_seed
    from unsloth import FastModel
    from unsloth.chat_templates import get_chat_template
    set_seed(SEED)
    model, processor = FastModel.from_pretrained(model_name=str(SELECTED_CHECKPOINT), max_seq_length=MAX_SEQUENCE_LENGTH, load_in_4bit=True)
    processor = get_chat_template(processor, chat_template='gemma-3')
    FastModel.for_inference(model)
    MODEL_INFO = {'executed': True, 'model_id': MODEL_ID, 'model_revision': MODEL_REVISION, 'selected_checkpoint': SELECTED_CHECKPOINT.name, 'adapter_model_sha256': EXPECTED_ADAPTER_MODEL_SHA256, 'max_sequence_length': MAX_SEQUENCE_LENGTH, 'load_in_4bit': True, 'adapter_merged': False}
print(json.dumps(MODEL_INFO, indent=2))


## 6. Frozen deterministic 25-record generation

The record selection is label-independent and frozen in `docs/f2_p1_private_dev_smoke_protocol.md`. Generation uses `do_sample=False`, no temperature argument, and `max_new_tokens=256`. Raw and parsed responses are written privately and never printed. The public summary deliberately omits the private development exact-match count.


In [ ]:
if RUN_PRIVATE_DEV_SMOKE:
    from collections import Counter
    from scripts.nahw_baseline_utils import parse_model_response
    order = sorted(range(len(private_dev)), key=lambda index: hashlib.sha256(f"F2-P1-dev-smoke|{SEED}|{private_dev[index]['record_id']}".encode('utf-8')).hexdigest())[:25]
    selected_id_bytes = ''.join(private_dev[index]['record_id'] + '\n' for index in order).encode('utf-8')
    selected_record_ids_sha256 = hashlib.sha256(selected_id_bytes).hexdigest()
    OUTPUT_DIR = Path('/kaggle/working/f2_p1_private_dev_smoke')
    if OUTPUT_DIR.exists():
        raise RuntimeError('Private smoke output directory already exists; refusing to overwrite it.')
    OUTPUT_DIR.mkdir(parents=True)
    predictions_path = OUTPUT_DIR / 'dev_predictions.jsonl'
    warning_counts = Counter(); exact_matches = 0; empty_outputs = 0; maximum_input_tokens = 0
    with predictions_path.open('w', encoding='utf-8') as output_stream:
        for position, index in enumerate(order, 1):
            row = private_dev[index]
            inference_messages = [{'role': message['role'], 'content': [{'type': 'text', 'text': message['content']}]} for message in row['prompt']]
            inputs = processor.apply_chat_template(inference_messages, tokenize=True, add_generation_prompt=True, return_tensors='pt', return_dict=True).to('cuda')
            input_tokens = int(inputs['input_ids'].shape[-1]); maximum_input_tokens = max(maximum_input_tokens, input_tokens)
            if input_tokens > MAX_SEQUENCE_LENGTH:
                raise RuntimeError('A selected development prompt exceeds the frozen token limit; refusing truncation.')
            with torch.inference_mode():
                generated = model.generate(**inputs, do_sample=False, max_new_tokens=256)
            response = processor.decode(generated[0, inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
            parsed, warnings = parse_model_response(response)
            gold = row['completion'][0]['content']
            exact_match = parsed == gold
            exact_matches += int(exact_match); empty_outputs += int(not parsed); warning_counts.update(warnings)
            private_result = {'record_id': row['record_id'], 'prompt_sha256': hashlib.sha256(row['prompt'][0]['content'].encode('utf-8')).hexdigest(), 'gold_sha256': hashlib.sha256(gold.encode('utf-8')).hexdigest(), 'raw_response': response, 'parsed_response': parsed, 'parsing_warnings': warnings, 'exact_match': exact_match}
            output_stream.write(json.dumps(private_result, ensure_ascii=False, sort_keys=True) + '\n')
            print({'completed': position, 'of': 25})
    predictions_sha256 = sha256_file(predictions_path)
    private_summary = {'protocol_id': 'F2-P1', 'gate': 'private_25_record_development_smoke', 'completed': True, 'private_exact_match_count': exact_matches, 'private_exact_match_metric_publication_allowed': False, 'predictions_sha256': predictions_sha256, 'contains_corpus_text': False}
    private_summary_path = OUTPUT_DIR / 'f2_p1_dev_smoke_private_summary.json'
    private_summary_path.write_text(json.dumps(private_summary, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    gate_status = 'passed' if empty_outputs == 0 else 'failed'
    public_summary = {'protocol_id': 'F2-P1', 'gate': 'private_25_record_development_smoke', 'gate_status': gate_status, 'execution': {'platform': 'Kaggle', 'repository_commit': ACTUAL_WORKFLOW_COMMIT, 'approval_reference': APPROVAL_REFERENCE}, 'checkpoint': {'selected_checkpoint': SELECTED_CHECKPOINT.name, 'model_revision': MODEL_REVISION, 'adapter_model_sha256': EXPECTED_ADAPTER_MODEL_SHA256, 'adapter_config_sha256': EXPECTED_ADAPTER_CONFIG_SHA256, 'checkpoint_selection_sha256': EXPECTED_SELECTION_SHA256, 'adapter_merged': False}, 'records': {'expected': 25, 'completed': 25, 'selected_record_ids_sha256': selected_record_ids_sha256, 'private_predictions_sha256': predictions_sha256, 'maximum_input_tokens': maximum_input_tokens}, 'decoding': {'do_sample': False, 'temperature_argument': None, 'max_new_tokens': 256, 'seed': SEED}, 'parsing': {'empty_output_count': empty_outputs, 'warning_counts': dict(sorted(warning_counts.items()))}, 'runtime': RUNTIME, 'safeguards': {'contains_corpus_text': False, 'private_development_metric_published': False, 'checkpoint_selection_changed': False, 'qalb_test_used': False, 'nahw_passage_used': False, 'training_executed_in_smoke_kernel': False, 'f3_executed': False, 'safety_diagnostics_executed': False, 'xg_executed': False}}
    public_summary_path = OUTPUT_DIR / 'f2_p1_dev_smoke_public_summary.json'
    public_summary_path.write_text(json.dumps(public_summary, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    print(json.dumps(public_summary, indent=2))
    if gate_status != 'passed':
        raise RuntimeError('Private development smoke produced an empty output; authorization is consumed and no retry is allowed.')
else:
    print('Private development smoke is disabled; no model load or generation was executed.')


## 7. Preserve private artifacts and export the text-free summary

Keep `dev_predictions.jsonl`, `f2_p1_dev_smoke_private_summary.json`, the selected adapter, and full logs private. Download only `f2_p1_dev_smoke_public_summary.json` for repository audit. Do not publish the private exact-match count. Preserve the first terminal run and do not retry without fresh authorization.
